# 02 — CNN from Scratch

Train a custom CNN with 5 convolutional blocks.
- Conv2d → BN → ReLU → Conv2d → BN → ReLU → MaxPool (×5)
- Global Average Pooling → FC classifier
- Data augmentation: flip, rotation, jitter, crop

### Expected Output
- Best checkpoint → `ml/artifacts/checkpoints/cnn_best.pth`
- Training curves, confusion matrix, per-class F1
- Classification report and training summary JSON

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if 'notebooks' in str(PROJECT_ROOT):
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/ajitkumarsingh/Desktop/desktop/cattle-breed-classifier-webapp


In [2]:
from ml.src.utils.seed import set_seed
from ml.src.utils.device import get_device
from ml.src.utils.io import load_config
from ml.src.utils.manifests import create_dataloaders
from ml.src.data.transforms import get_train_transforms, get_eval_transforms
from ml.src.models.cnn import CattleCNN
from ml.src.training.trainer import Trainer

set_seed(42)
device = get_device()

Using Apple MPS (Metal Performance Shaders)


## 1. Load Config & Data

In [3]:
config = load_config('cnn', PROJECT_ROOT / 'ml' / 'configs')
config['num_classes'] = 26

img_size = config['image']['size']
batch_size = config['training']['batch_size']

print(f'Image size: {img_size}')
print(f'Batch size: {batch_size}')
print(f'Conv channels: {config["model"]["architecture"]["conv_channels"]}')
print(f'Epochs: {config["training"]["num_epochs"]}')

Image size: 224
Batch size: 32
Conv channels: [32, 64, 128, 256, 512]
Epochs: 50


In [4]:
train_transforms = get_train_transforms(img_size=img_size)
eval_transforms = get_eval_transforms(img_size=img_size)

dataloaders = create_dataloaders(
    manifests_dir=PROJECT_ROOT / 'ml' / 'artifacts' / 'manifests',
    data_root=PROJECT_ROOT / 'Cattle_Resized',
    train_transform=train_transforms,
    eval_transform=eval_transforms,
    batch_size=batch_size,
    num_workers=config['data'].get('num_workers', 4),
)

class_names = dataloaders['train'].dataset.classes
print(f'Classes: {len(class_names)}')
print(f'Train: {len(dataloaders["train"].dataset)} | Val: {len(dataloaders["val"].dataset)} | Test: {len(dataloaders["test"].dataset)}')

Classes: 26
Train: 2139 | Val: 458 | Test: 459


## 2. Create Model

In [5]:
model = CattleCNN.from_config(config)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')
print(f'Model size: {total_params * 4 / 1024 / 1024:.1f} MB (float32)')

CattleCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runn

## 3. Train

In [6]:
trainer = Trainer(
    model=model,
    config=config,
    dataloaders=dataloaders,
    class_names=class_names,
    device=device,
    model_name='cnn',
)

training_summary = trainer.train()


Training: cnn
Device: mps
Epochs: 50



/Users/ajitkumarsingh/miniforge3/envs/cattle-classifier/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  Epoch   1 | Train Loss: 4.9079 | Train Acc: 0.0554 | Val Loss: 3.5658 | Val Acc: 0.0873 | LR: 0.001000
  Checkpoint saved: cnn_best.pth (value: 3.5658)
  Epoch   2 | Train Loss: 4.1389 | Train Acc: 0.0824 | Val Loss: 3.4800 | Val Acc: 0.0961 | LR: 0.000999
  Checkpoint saved: cnn_best.pth (value: 3.4800)
  Epoch   3 | Train Loss: 3.8920 | Train Acc: 0.0923 | Val Loss: 3.9241 | Val Acc: 0.1201 | LR: 0.000996
  EarlyStopping: 1/10
  Epoch   4 | Train Loss: 3.7722 | Train Acc: 0.0895 | Val Loss: 3.0622 | Val Acc: 0.1179 | LR: 0.000991
  Checkpoint saved: cnn_best.pth (value: 3.0622)
  Epoch   5 | Train Loss: 3.6928 | Train Acc: 0.1089 | Val Loss: 3.0745 | Val Acc: 0.1092 | LR: 0.000984
  EarlyStopping: 1/10
  Epoch   6 | Train Loss: 3.5790 | Train Acc: 0.1127 | Val Loss: 2.9818 | Val Acc: 0.1201 | LR: 0.000976
  Checkpoint saved: cnn_best.pth (value: 2.9818)
  Epoch   7 | Train Loss: 3.5056 | Train Acc: 0.1165 | Val Loss: 3.0962 | Val Acc: 0.1201 | LR: 0.000965
  EarlyStopping: 1/10
  E

python(85066) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(85068) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(85069) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(85070) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/ajitkumarsingh/miniforge3/envs/cattle-classifier/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/ajitkumarsingh/miniforge3/envs/cattle-classifier/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ajitkumarsingh/miniforge3/envs/cattle-classifier/lib/python3.12/mu

RuntimeError: DataLoader worker (pid(s) 85066, 85069, 85070) exited unexpectedly

## 4. Evaluate on Test Set

In [7]:
test_results = trainer.evaluate(split='test')


Evaluating: cnn on test

Loaded best checkpoint from epoch 24


python(96188) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(96189) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(96190) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(96191) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



TEST Results:
  Loss:          2.6294
  Accuracy:      0.2004
  Macro F1:      0.0875
  Macro Prec:    0.0788
  Macro Recall:  0.1186
  Avg Latency:   13.84 ms
  Model Size:    18.52 MB


## 5. Save Artifacts

In [ ]:
trainer.save_artifacts()

print('\n=== CNN from Scratch Summary ===')
print(f'Best Val Loss:  {training_summary["best_val_loss"]:.4f}')
print(f'Best Val Acc:   {training_summary["best_val_accuracy"]:.4f}')
print(f'Test Accuracy:  {test_results["metrics"]["accuracy"]:.4f}')
print(f'Test Macro F1:  {test_results["metrics"]["macro_f1"]:.4f}')
print(f'Latency:        {test_results["latency"]["avg_ms"]:.2f} ms')
print(f'Model Size:     {test_results["model_size_mb"]:.2f} MB')

## 6. Classification Report

In [8]:
print(test_results['metrics']['classification_report'])

                    precision    recall  f1-score   support

      Alambadi Cow       0.08      0.07      0.08        14
    Amritmahal Cow       0.00      0.00      0.00        14
     Banni Buffalo       0.00      0.00      0.00         5
        Bargur Cow       0.00      0.00      0.00         9
         Dangi Cow       0.00      0.00      0.00        12
         Deoni Cow       0.00      0.00      0.00        16
           Gir Cow       0.25      0.50      0.33        38
      Hallikar Cow       0.16      0.29      0.20        28
Jaffrabadi Buffalo       0.08      0.06      0.07        16
      Kangayam Cow       0.00      0.00      0.00        18
       Kankrej Cow       0.14      0.04      0.06        27
     Kasaragod Cow       0.00      0.00      0.00        14
      Kenkatha Cow       0.00      0.00      0.00         8
     Kherigarh Cow       0.00      0.00      0.00         5
  Malnad gidda Cow       0.27      0.25      0.26        16
   Mehsana Buffalo       0.14      0.20

---
**✅ CNN from Scratch complete.** Proceed to `03_resnet_transfer_learning.ipynb`.